# Reading Data — Demo

**Course:** Machine Learning Foundation  
**Topic:** Membaca data dari database SQLite dengan Pandas.

Notebook ini merupakan penulisan ulang untuk pembelajaran dan portofolio. Database sumber disertakan di folder `data/`.

## 1. Tujuan pembelajaran
- Menghubungkan Python dengan database SQLite.
- Membaca tabel menggunakan SQL dan Pandas.
- Mengelompokkan data dengan `GROUP BY`, menghitung agregat, dan mengurutkan hasil.
- Membaca hasil dalam beberapa bagian (*chunks*).

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("data/classic_rock.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database tidak ditemukan: {DB_PATH.resolve()}")

connection = sqlite3.connect(DB_PATH)
print("Koneksi SQLite berhasil.")

Koneksi SQLite berhasil.


## 2. Melihat tabel yang tersedia

Tabel `sqlite_master` menyimpan metadata database, termasuk nama tabel.

In [2]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    connection
)
tables

,name
0,rock_plays
1,rock_songs


## 3. Membaca semua data lagu

`read_sql_query()` menjalankan query SQL dan mengembalikan hasilnya sebagai DataFrame.

In [3]:
songs = pd.read_sql_query("SELECT * FROM rock_songs;", connection)
print("Ukuran data:", songs.shape)
display(songs.head())

Ukuran data: (1650, 4)


,Song,Artist,Release_Year,PlayCount
0,Caught Up in You,.38 Special,1982.0,82
1,Hold On Loosely,.38 Special,1981.0,85
2,Rockin' Into the Night,.38 Special,1980.0,18
3,Art For Arts Sake,10cc,1975.0,1
4,Kryptonite,3 Doors Down,2000.0,13


## 4. Mengambil ringkasan per artis dan tahun rilis

Query ini menghitung jumlah lagu dan rata-rata jumlah pemutaran untuk setiap kombinasi artis dan tahun.

In [4]:
summary_query = """
SELECT Artist, Release_Year,
       COUNT(*) AS num_songs,
       AVG(PlayCount) AS avg_plays
FROM rock_songs
GROUP BY Artist, Release_Year
ORDER BY num_songs DESC, avg_plays DESC;
"""
summary = pd.read_sql_query(summary_query, connection)
display(summary.head(10))

,Artist,Release_Year,num_songs,avg_plays
0,The Beatles,1967.0,23,6.565217
1,Led Zeppelin,1969.0,18,21.000000
2,The Beatles,1965.0,15,3.800000
3,The Beatles,1969.0,13,15.000000
4,The Beatles,1968.0,13,13.000000
5,Led Zeppelin,1975.0,12,14.166667
6,Led Zeppelin,1970.0,12,13.166667
7,Pink Floyd,1979.0,11,41.454545
8,Pink Floyd,1973.0,10,29.100000
9,The Doors,1967.0,10,28.900000


## 5. Membaca hasil secara bertahap

Parameter `chunksize` menghasilkan iterator berisi DataFrame kecil. Ini bermanfaat untuk tabel yang terlalu besar untuk dimuat sekaligus. `Release_Year` adalah tahun, bukan tanggal lengkap, jadi kita tidak mengubahnya menjadi tipe tanggal.

In [5]:
chunk_query = """
SELECT Artist, Release_Year, COUNT(*) AS num_songs, AVG(PlayCount) AS avg_plays
FROM rock_songs
GROUP BY Artist, Release_Year
ORDER BY num_songs DESC;
"""
for chunk_number, chunk in enumerate(pd.read_sql_query(chunk_query, connection, chunksize=5), start=1):
    print(f"Chunk {chunk_number}")
    display(chunk)
    if chunk_number == 3:
        break

Chunk 1


,Artist,Release_Year,num_songs,avg_plays
0,The Beatles,1967.0,23,6.565217
1,Led Zeppelin,1969.0,18,21.000000
2,The Beatles,1965.0,15,3.800000
3,The Beatles,1968.0,13,13.000000
4,The Beatles,1969.0,13,15.000000


Chunk 2


,Artist,Release_Year,num_songs,avg_plays
0,Led Zeppelin,1970.0,12,13.166667
1,Led Zeppelin,1975.0,12,14.166667
2,Pink Floyd,1979.0,11,41.454545
3,Pink Floyd,1973.0,10,29.100000
4,The Doors,1967.0,10,28.900000


Chunk 3


,Artist,Release_Year,num_songs,avg_plays
0,Fleetwood Mac,1977.0,9,35.666667
1,Jimi Hendrix,1967.0,9,24.888889
2,The Beatles,1963.0,9,2.444444
3,The Beatles,1964.0,9,3.111111
4,Elton John,1973.0,8,18.500000


## 6. Kesimpulan

- SQLite dapat diakses dari Python dengan `sqlite3`.
- Pandas memudahkan hasil SQL menjadi DataFrame.
- SQL dapat melakukan penyaringan, pengelompokan, agregasi, dan pengurutan sebelum data dianalisis lebih lanjut.
- `chunksize` membantu memproses hasil query secara bertahap.

In [6]:
connection.close()
print("Koneksi database ditutup.")

Koneksi database ditutup.
